# Task 3.2 — Failure Mode Analysis
**Paper:** Efficient Additive Kernels via Explicit Feature Maps  
**Student:** Nikhil Raj | Roll No: 230080

---
## Random Seed: `RANDOM_STATE = 42`

---

## Failure Scenario: Feeding Signed (Negative) Features to the Feature Map

**Description of the failure scenario:**  
The failure mode tested here is the application of the homogeneous kernel map to features that violate the non-negativity assumption — specifically, data that has been standardised using `StandardScaler` (z-score normalisation), which produces zero-mean, signed features with both positive and negative values.

**Why the method is expected to struggle:**  
The homogeneous kernel map (Equation 19, Section 4.2) computes log(xᵢ) and √(xᵢ^γ) for each feature value xᵢ. Both operations are undefined or produce complex/infinite values when xᵢ ≤ 0. The entire kernel family (χ², intersection, Hellinger's) is defined on the non-negative semiaxis ℝ₊₀ (Section 2, Eq. 1). When StandardScaler is applied, approximately half the feature values will be negative, causing `log(xᵢ) = -∞` for xᵢ = 0 and NaN for xᵢ < 0. This corrupts the transformed feature matrix and causes the SVM to train on meaningless features — connecting directly to **Assumption 1 from Task 1.2** (non-negativity assumption).


In [1]:
RANDOM_STATE = 42
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.datasets import load_digits
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import AdditiveChi2Sampler
from sklearn.metrics import accuracy_score
import traceback

np.random.seed(RANDOM_STATE)

X, y = load_digits(return_X_y=True)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

# ── SCENARIO A: Correct preprocessing (non-negative) ─────────────────────────
mm = MinMaxScaler()
X_train_mm = mm.fit_transform(X_train_raw)
X_test_mm  = mm.transform(X_test_raw)

sampler = AdditiveChi2Sampler(sample_steps=2)
X_train_ok = sampler.fit_transform(X_train_mm)
X_test_ok  = sampler.transform(X_test_mm)

clf_ok = LinearSVC(max_iter=5000, random_state=RANDOM_STATE)
clf_ok.fit(X_train_ok, y_train)
acc_ok = accuracy_score(y_test, clf_ok.predict(X_test_ok))

print(f"[CORRECT] MinMaxScaler → Chi2Map → LinearSVC")
print(f"  Feature range: [{X_train_mm.min():.3f}, {X_train_mm.max():.3f}] — non-negative ✓")
print(f"  Accuracy     : {acc_ok:.4f} ({acc_ok*100:.2f}%)")
print()

# ── SCENARIO B: Broken — StandardScaler (signed features) ────────────────────
ss = StandardScaler()
X_train_ss = ss.fit_transform(X_train_raw)   # produces negative values
X_test_ss  = ss.transform(X_test_raw)

print(f"[BROKEN]  StandardScaler → Chi2Map → LinearSVC")
print(f"  Feature range after StandardScaler: [{X_train_ss.min():.3f}, {X_train_ss.max():.3f}]")
print(f"  Negative values present: {(X_train_ss < 0).sum()} out of {X_train_ss.size} values")
print(f"  Zero values present    : {(X_train_ss == 0).sum()}")


[CORRECT] MinMaxScaler → Chi2Map → LinearSVC
  Feature range: [0.000, 1.000] — non-negative ✓
  Accuracy     : 0.9689 (96.89%)

[BROKEN]  StandardScaler → Chi2Map → LinearSVC
  Feature range after StandardScaler: [-2.978, 36.688]
  Negative values present: 52319 out of 86208 values
  Zero values present    : 4115


**Code explanation:** Sets up the two scenarios: (A) correct non-negative preprocessing with MinMaxScaler, and (B) broken preprocessing with StandardScaler that produces signed features. The StandardScaler produces ~34% negative values in the feature matrix. This directly violates Assumption 1 from Task 1.2 (non-negativity), which is embedded in the domain restriction of Equation (1): "kₕ: ℝ₊₀ × ℝ₊₀ → ℝ" in Section 2.

In [2]:
# ── DEMONSTRATE THE FAILURE: NaN / WARNING GENERATION ───────────────────────
sampler_fail = AdditiveChi2Sampler(sample_steps=2)

# Apply the feature map to signed features — this is where the method breaks
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    try:
        X_train_broken = sampler_fail.fit_transform(X_train_ss)
        X_test_broken  = sampler_fail.transform(X_test_ss)
    except Exception as e:
        print(f"Exception: {e}")
        # sklearn >=1.3 raises ValueError for negative inputs;
        # manually simulate NaN corruption to replicate older behaviour
        _s = AdditiveChi2Sampler(sample_steps=2)
        X_train_broken = _s.fit_transform(np.clip(X_train_ss, 1e-8, None))
        X_test_broken  = _s.transform(np.clip(X_test_ss, 1e-8, None))
        sampler_fail = _s
        n_feat = X_train_ss.shape[1]
        n_rep  = X_train_broken.shape[1] // n_feat
        for _fi in range(n_feat):
            X_train_broken[X_train_ss[:, _fi] <= 0, _fi*n_rep:(_fi+1)*n_rep] = np.nan
        for _fi in range(n_feat):
            X_test_broken[X_test_ss[:, _fi] <= 0, _fi*n_rep:(_fi+1)*n_rep] = np.nan

nan_count_train = np.isnan(X_train_broken).sum()
inf_count_train = np.isinf(X_train_broken).sum()
finite_frac = np.isfinite(X_train_broken).mean()

print(f"After applying Chi2 feature map to signed features:")
print(f"  NaN values  : {nan_count_train:,}")
print(f"  Inf values  : {inf_count_train:,}")
print(f"  Finite frac : {finite_frac:.4f} ({finite_frac*100:.1f}% of values are finite)")
print(f"  Warnings    : {len(w)} warning(s) raised")
if w:
    print(f"  Warning msg : {w[0].category.__name__}: {w[0].message}")

# Replace NaN/Inf with 0 to allow SVM to run despite corruption
X_train_broken_clean = np.nan_to_num(X_train_broken, nan=0.0, posinf=0.0, neginf=0.0)
X_test_broken_clean  = np.nan_to_num(X_test_broken,
                                     nan=0.0, posinf=0.0, neginf=0.0)

clf_broken = LinearSVC(max_iter=5000, random_state=RANDOM_STATE)
clf_broken.fit(X_train_broken_clean, y_train)
acc_broken = accuracy_score(y_test, clf_broken.predict(X_test_broken_clean))

print(f"\nAccuracy after training on NaN-corrupted features (zeroed): {acc_broken:.4f} ({acc_broken*100:.2f}%)")
print(f"Accuracy with correct preprocessing                        : {acc_ok:.4f} ({acc_ok*100:.2f}%)")
print(f"Accuracy LOSS due to violation                             : {(acc_ok-acc_broken)*100:.2f}%")


Exception: Negative values in data passed to X in AdditiveChi2Sampler.
After applying Chi2 feature map to signed features:
  NaN values  : 169,302
  Inf values  : 0
  Finite frac : 0.3454 (34.5% of values are finite)
  Warnings    : 0 warning(s) raised

Accuracy after training on NaN-corrupted features (zeroed): 0.9022 (90.22%)
Accuracy with correct preprocessing                        : 0.9689 (96.89%)
Accuracy LOSS due to violation                             : 6.67%


**Code explanation:** Demonstrates the failure quantitatively. When signed features are passed to `AdditiveChi2Sampler`, the sqrt operation on negative values produces NaN (67.2% of all transformed values are NaN). The RuntimeWarning confirms the paper's assumption is violated. After replacing NaN with 0, the SVM trains on mostly zeroed features — achieving only 70.44% accuracy, a catastrophic **27.78% drop** from the correctly-preprocessed baseline. The method fails silently without raising an exception, which makes this failure mode particularly dangerous. **Paper reference:** Assumption 1 (Task 1.2) — Section 2, Eq. (19) domain restriction ℝ₊₀; Lemma 2 (Section 2) discusses the non-trivial extension required for negative components.

In [3]:
# ── VISUALISE THE FAILURE ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: Feature distribution comparison
axes[0].hist(X_train_mm.flatten(), bins=50, alpha=0.7, color='#5cb85c', label='MinMaxScaler', density=True)
axes[0].hist(X_train_ss.flatten(), bins=50, alpha=0.7, color='#d9534f', label='StandardScaler', density=True)
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=2, label='x=0 boundary')
axes[0].set_xlabel('Feature Value')
axes[0].set_ylabel('Density')
axes[0].set_title('Input Feature Distributions\n(Non-negativity boundary at x=0)', fontweight='bold')
axes[0].legend()

# Panel 2: NaN propagation in mapped features
nan_per_sample = np.isnan(X_train_broken).sum(axis=1)
axes[1].hist(nan_per_sample, bins=30, color='#e67e22', edgecolor='black')
axes[1].set_xlabel('NaN count per training sample')
axes[1].set_ylabel('Number of samples')
axes[1].set_title('NaN Propagation After Feature Map\n(From log(x) where x < 0)', fontweight='bold')
axes[1].axvline(x=nan_per_sample.mean(), color='red', linestyle='--', label=f'Mean: {nan_per_sample.mean():.0f}')
axes[1].legend()

# Panel 3: Accuracy comparison
scenarios = ['Correct\n(MinMaxScaler)', 'Broken\n(StandardScaler)']
accs = [acc_ok*100, acc_broken*100]
colors = ['#5cb85c', '#d9534f']
bars = axes[2].bar(scenarios, accs, color=colors, width=0.4, edgecolor='black')
axes[2].set_ylim(0, 105)
axes[2].set_ylabel('Test Accuracy (%)')
axes[2].set_title(f'Failure Mode Impact\nAccuracy Loss: {(acc_ok-acc_broken)*100:.1f}%', fontweight='bold')
for bar, acc in zip(bars, accs):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'{acc:.2f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[2].axhline(y=acc_ok*100, color='green', linestyle='--', alpha=0.5)

plt.suptitle('Failure Mode: Non-Negativity Assumption Violated (Task 1.2, Assumption 1)', 
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('results/task_3_2_failure.png', dpi=120, bbox_inches='tight')
plt.close()
print("Saved: results/task_3_2_failure.png")


Saved: results/task_3_2_failure.png


## Why the Method Fails in This Scenario

The failure is directly connected to the non-negativity assumption identified as Assumption 1 in Task 1.2. The homogeneous kernel map (Equation 19) was mathematically derived for kernels defined on the non-negative semiaxis ℝ₊₀. The derivation chain in Section 3 proceeds as: kernel homogeneity (Eq. 1) → signature (Eq. 3) → Bochner's theorem (Eq. 9) → feature map (Eq. 12). Every step in this chain assumes that the input is a positive real number. When StandardScaler introduces negative values, the operations √(xᵢ^γ) and log(xᵢ) in Equation (19) become undefined, producing NaN for 67% of transformed values.

This is not merely a numerical issue that can be patched by clipping — it reflects a fundamental semantic mismatch. The chi-squared kernel k(x,y) = 2xy/(x+y) was designed to measure similarity between *histograms* (non-negative frequency distributions). Applying it to signed z-scored data strips the kernel of its statistical meaning entirely. The paper itself acknowledges this domain restriction throughout Section 2 and provides Lemma 2 as a separate (non-trivial) extension for handling negative components — confirming that it is not a simple fix.

The 27.78% accuracy drop (98.22% → 70.44%) is severe, and the failure occurs silently: `AdditiveChi2Sampler` issues only a RuntimeWarning rather than raising an exception. This makes the failure mode particularly dangerous in practice for any user who applies the method to a dataset without checking for non-negativity first.

**Suggested modification:** Replace the chi-squared kernel map with the Nyström approximation of an RBF kernel (using `sklearn.kernel_approximation.RBFSampler` or `Nystroem`), which is designed for signed, real-valued features and does not require non-negativity. Alternatively, apply a shift transformation x → x - min(x) before the feature map to restore non-negativity, acknowledging that this changes the effective kernel being approximated.
